In [1]:
import pandas as pd
import ast
import sys
import os
import numpy as np

In [2]:
from server_config import DATA_PATH
sys.path.append(os.path.abspath("functions"))

from ML_Pipeline import  MatchConfig, run_pipeline
from therapist_clustering import repair_therapist_age

### 1. Therapist data

### Data exploration

In [3]:
df_all = pd.read_csv(DATA_PATH + "/toni_preprocessed.csv")

In [4]:
df_all.therapist_id.nunique()

174

In [5]:
df_all.groupby("therapist_id")["patient_id"].nunique().describe()

count    174.000000
mean       5.494253
std        3.721127
min        1.000000
25%        3.000000
50%        4.000000
75%        8.000000
max       21.000000
Name: patient_id, dtype: float64

In [6]:
df_all.patient_id.nunique()

956

### 2.2 Run Model

In [7]:
# Matrix Factorization

In [8]:
from ML_Pipeline import MatchConfig, run_pipeline

cfg = MatchConfig(
    # --- IDs & timing ---
    PATIENT_ID_COL="patient_id",
    THERAPIST_ID_COL="therapist_id",
    DATE_COL="u_registrationDate_p",
    DATE_DAYFIRST=True,    # set False if your dates are like 2024-03-31
    DATE_FORMAT=None,      # or e.g. "%Y-%m-%d" if you need it

    # --- Target ---
    LABEL_COL="change_distress",   # your precomputed delta
    TARGET_MODE="delta",           # tells the pipeline "higher is better", alternative is t1
    STANDARDIZE_Y=True,            # keeps models comparably scaled

    # --- Features ---
    patient_numeric=['t0_ses_p', 'surveys.t0.answers.1585.discrimination_1', 'surveys.t0.answers.1585.discrimination_2',
         'surveys.t0.answers.1585.discrimination_3', 'surveys.t0.answers.1585.discrimination_4', 'surveys.t0.answers.1585.discrimination_5',
          't0_phq_p_1', 't0_phq_p_2', 't0_phq_p_3', 't0_phq_p_4', 't0_phq_p_5', 't0_phq_p_6', 't0_phq_p_7', 't0_phq_p_8',
         't0_gad_p_1', 't0_gad_p_2', 't0_gad_p_3', 't0_gad_p_4', 't0_gad_p_5', 't0_gad_p_6', 't0_gad_p_7','t0_swls_p_1',
         't0_swls_p_2', 't0_swls_p_3', 't0_swls_p_4', 't0_swls_p_5', 't0_aqol_p_1','t0_aqol_p_2', 't0_aqol_p_3',
         't0_aqol_p_4', 't0_aqol_p_5', 't0_aqol_p_6', 't0_aqol_p_7', 't0_aqol_p_8', 't0_aqol_p_9', 't0_aqol_p_10',
         't0_aqol_p_11', 't0_aqol_p_12', 't0_aqol_p_13', 't0_aqol_p_14', 't0_aqol_p_15', 't0_aqol_p_16', 't0_aqol_p_17',
         't0_aqol_p_18', 't0_aqol_p_19', 't0_aqol_p_20', 't0_aqol_p_21', 't0_aqol_p_22', 't0_aqol_p_23', 't0_aqol_p_24',
         't0_aqol_p_25', 't0_aqol_p_26', 't0_aqol_p_27', 't0_aqol_p_28', 't0_aqol_p_29', 't0_aqol_p_30', 't0_aqol_p_31',
         't0_aqol_p_32', 't0_aqol_p_33', 't0_aqol_p_34', 't0_aqol_p_35', 't0_audit_p_1', 't0_audit_p_2', 't0_audit_p_3',
        't0_dudit_p_1', 't0_dudit_p_2', 't0_dudit_p_3', 't0_dudit_p_4', 't0_msfq_p_1', 't0_msfq_p_2', 't0_msfq_p_3',
         't0_msfq_p_4', 't0_msfq_p_5', 't0_pid_p_1', 't0_pid_p_2', 't0_pid_p_3', 't0_pid_p_4', 't0_pid_p_5', 't0_pid_p_6',
        't0_pid_p_7', 't0_pid_p_8', 't0_pid_p_9', 't0_pid_p_10', 't0_pid_p_11', 't0_pid_p_12', 't0_pid_p_13', 't0_pid_p_14',
         't0_pid_p_15', 't0_pid_p_16', 't0_pid_p_17', 't0_pid_p_18', 't0_pid_p_19', 't0_pid_p_20', 't0_pid_p_21',
         't0_pid_p_22', 't0_pid_p_23', 't0_pid_p_24', 't0_pid_p_25', 't0_pid_p_26', 't0_pid_p_27', 't0_pid_p_28',
         't0_pid_p_29', 't0_pid_p_30', 't0_pid_p_31', 't0_pid_p_32', 't0_pid_p_33', 't0_pid_p_34', 't0_pid_p_35',
         't0_pid_p_36', 't0_opd_p_1', 't0_opd_p_2', 't0_opd_p_3', 't0_opd_p_4', 't0_opd_p_5', 't0_opd_p_6', 't0_opd_p_7',
         't0_opd_p_8', 't0_opd_p_9', 't0_opd_p_10', 't0_opd_p_11', 't0_opd_p_12', 't0_ace_p_1', 't0_ace_p_2',
         't0_ace_p_3', 't0_ace_p_4', 't0_ace_p_5', 't0_ace_p_6', 't0_ace_p_7', 't0_ace_p_8', 't0_ace_p_9', 't0_ace_p_10',
         't0_ask_p_1', 't0_ask_p_2', 't0_ask_p_3', 't0_mhse_p_1', 't0_mhse_p_2', 't0_mhse_p_3', 't0_mhse_p_4', 't0_mhse_p_5',
         't0_mhse_p_6', 't0_change_p_1', 't0_change_p_2', 't0_dropout_p', 't0_phq8_p_sum', 't0_gad7_p_sum', 't0_swls_p_sum',
         't0_aqol_p_coping','t0_aqol_p_relationship', 't0_aqol_p_independent_living', 't0_aqol_p_mental_health',
         't0_aqol_p_pain', 't0_aqol_p_self_worth', 't0_aqol_p_happiness', 't0_aqol_p_senses', 't0_audit_p_sum',
         't0_dudit_p_sum', 't0_msfq_p_sum', 't0_pid_p_negative_affect', 't0_pid_p_antagonism', 't0_pid_p_disinhibition',
         't0_pid_p_detachment', 't0_pid_p_psychoticism', 't0_pid_p_anankastia', 't0_opd_p_self', 't0_opd_p_contact',
         't0_opd_p_relationship', 't0_ace_p_sum', 't0_ask_p_sum', 't0_mhse_p_sum','t0_age_p'],
    patient_categorical=[
        't0_gender_p','t0_school_p','t0_employment_p','t0_location_p',
        't0_therapy_p_1','t0_therapy_p_2','t0_familyhistory_p','t0_meds_p',
        't0_relationship_p','t0_migration_p_1','t0_migration_p_2','t0_migration_p_3',
        't0_disease_p_A','t0_disease_p_B','t0_disease_p_C','t0_disease_p_D',
        't0_disease_p_E','t0_disease_p_F','t0_disease_p_G',
    ],
    therapist_numeric=[
        't0_age_t','t0_experience_t','t0_bfi_t_1','t0_bfi_t_2','t0_bfi_t_3','t0_bfi_t_4','t0_bfi_t_5',
        't0_bfi_t_6','t0_bfi_t_7','t0_bfi_t_8','t0_bfi_t_9','t0_bfi_t_10',
        't0_bfi_t_extraversion','t0_bfi_t_agreeableness','t0_bfi_t_conscientiousness',
        't0_bfi_t_neuroticism','t0_bfi_t_openness'
    ],
    therapist_categorical=[
        't0_method_t_CBT','t0_method_t_PDT','t0_method_t_ST','t0_method_t_AP',
        't0_appliedmethod_t_VT','t0_appliedmethod_t_TP','t0_appliedmethod_t_AP',
        't0_appliedmethod_t_ST','t0_appliedmethod_t_Other',
        't0_location_t','t0_gender_t','t0_relationship_t','t0_sexualorientation_t','t0_children_t'
    ],
    include_therapist_id_feature=False,  # set True to add therapist_id as a categorical feature

    # --- Imputation ---
    impute_strategy="median",  # or "iterative"

    # --- HPO grids (RMSE-tuned). You can leave defaults or tweak: ---
    fm_param_grid={
        "regressor__k": [8, 16, 32],
        "regressor__lr": [1e-3, 5e-4],
        "regressor__n_epochs": [40],
        "regressor__batch_size": [256],
        "regressor__weight_decay": [0.0, 1e-5],
    },
    ridge_param_grid={
        "regressor__alpha": [1e-3, 1e-2, 1e-1, 1, 10, 1e2]
    },
    cat_param_grid={
        "regressor__depth": [6, 8],
        "regressor__l2_leaf_reg": [3.0, 10.0],
        "regressor__learning_rate": [0.03, 0.06],
        "regressor__iterations": [400, 800,1000],

        
        # regularization
        "regressor__l2_leaf_reg":      [1.0, 3.0, 10.0, 30.0],
        "regressor__min_data_in_leaf": [1, 10, 30, 60]},

    
)


In [9]:
results = run_pipeline(df_all, cfg)

# Minimal, necessary summary metrics only (per model; best flagged in is_best)
results["summary"]


2025-09-24 10:34:00,480 [INFO] Match_Pipeline: orientation: TARGET_MODE=delta -> OUTCOME_HIGHER_IS_BETTER=True
2025-09-24 10:34:00,481 [INFO] Match_Pipeline: date: parsing 'u_registrationDate_p' (dayfirst=True, format=None)
2025-09-24 10:34:00,889 [INFO] Match_Pipeline: Therapists: 174 | Train: 655 | Val: 142 | Test: 159
2025-09-24 10:34:00,891 [INFO] Match_Pipeline: candidates: 174 therapists in train
2025-09-24 10:34:00,897 [INFO] Match_Pipeline: matrices: X_train=(655, 222), X_val=(142, 222)
2025-09-24 10:34:00,898 [INFO] Match_Pipeline: scoring: using RMSE (negated) on observed dyads.


/home/leha18/.conda/envs/torchenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2025-09-24 10:44:19,653 [INFO] Match_Pipeline: [Selected] cat | policy=test


,model,val_rmse,val_failed,test_rmse,ord_ll,ord_ll_per_case,ord_pseudo_r2,ord_coef_change_std,ord_or_change_std,ord_link,ord_optimizer,ord_sd_used
0,cat,32.065937,False,27.535724,-737.475106,-4.638208,NaN,-0.278442,0.756962,logit,lbfgs,28.990811
1,fm,32.449302,False,31.526588,-714.427555,-4.493255,NaN,0.060433,1.062297,logit,lbfgs,28.990811
2,ridge,34.968634,False,30.664450,-805.618312,-5.066782,NaN,0.108917,1.115070,logit,lbfgs,28.990811


In [10]:
# Aggregated therapist recommendation counts (top1/top5) for the chosen model
results["therapist_recommendations"].head(20)

,therapist_id,top1_count,top5_count,n_test_patients,top1_share,top5_share
0,188,26,55,159,0.163522,0.345912
1,938,25,60,159,0.157233,0.377358
2,32,9,20,159,0.056604,0.125786
3,522,8,56,159,0.050314,0.352201
4,160,6,46,159,0.037736,0.289308
5,51,6,19,159,0.037736,0.119497
6,146,5,14,159,0.031447,0.088050
7,33,4,17,159,0.025157,0.106918
8,852,4,12,159,0.025157,0.075472
9,100,4,7,159,0.025157,0.044025


In [11]:
# Optional: predictions for the first test patient (chosen model)
results["shap_interactions_ohe"].head(50)


,feat_i,feat_j,score,orig_i,orig_j
0,p_num__t0_phq_p_5,p_num__t0_gad_p_7,0.002658,NaN,NaN
1,p_num__t0_pid_p_15,p_num__t0_phq8_p_sum,0.002257,NaN,NaN
2,p_num__t0_phq_p_5,p_num__t0_opd_p_2,0.002033,NaN,NaN
3,p_num__t0_mhse_p_1,p_num__t0_phq8_p_sum,0.001978,NaN,NaN
4,p_num__t0_opd_p_5,p_num__t0_gad7_p_sum,0.001928,NaN,NaN
5,p_num__t0_pid_p_5,p_num__t0_gad7_p_sum,0.001592,NaN,NaN
6,p_num__t0_gad_p_5,p_num__t0_aqol_p_18,0.001568,NaN,NaN
7,p_num__t0_aqol_p_27,p_num__t0_gad7_p_sum,0.001541,NaN,NaN
8,p_num__t0_opd_p_2,p_num__t0_gad7_p_sum,0.001481,NaN,NaN
9,p_num__t0_pid_p_antagonism,p_num__t0_mhse_p_sum,0.001380,NaN,NaN


In [12]:
results["cat_importance_ohe"].head(50)

,feature,importance,orig_feature
0,p_num__t0_gad7_p_sum,12.233516,NaN
1,p_num__t0_phq8_p_sum,4.442696,NaN
2,p_num__t0_gad_p_2,3.741820,NaN
3,p_num__t0_change_p_1,3.046592,NaN
4,p_num__t0_gad_p_7,2.987212,NaN
5,p_num__t0_phq_p_5,2.819817,NaN
6,p_num__t0_phq_p_1,2.466015,NaN
7,p_num__t0_opd_p_2,2.451355,NaN
8,p_num__t0_gad_p_1,2.389846,NaN
9,p_num__t0_mhse_p_sum,2.289991,NaN


In [13]:
b

NameError: name 'b' is not defined

In [ ]:
# CatBoost 

In [ ]:
out["therapist_recommendation_counts"]